# 13. キャップストーン:1つの回帰、3つのレンズ(ベイズの視点)

線形代数編・ニューラルネット編と**まったく同じデータ**(`make_capstone_dataset`)を使う。

この章のレンズ:重み $w$ に事前分布 $\mathcal N(0, \sigma_w^2 I)$ を置き、**事後分布** $p(w \mid \Phi, y) = \mathcal N(\mu, \Sigma)$ を求める。事後平均 $\mu$ はちょうどリッジ解($\lambda = \sigma^2/\sigma_w^2$)、事後の**分散**が予測の不確実性になる。

```{admonition} 核心 — ひとことで
:class: tip
**同じ回帰を 3 つのレンズで見ると、リッジ解＝勾配降下の到達点＝ベイズ事後平均、と一致する。**
ガウス事前 × ガウス尤度 → ガウス事後。事後平均 $\mu$ がリッジ解（$\lambda=\sigma^2/\sigma_w^2$）、事後の**分散**が予測の不確実性。
線形代数は「解そのもの」、ニューラルネットは「解への辿り方」、ベイズは「解の周りの不確実性」を与える。
```

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import plotly.io as pio

from bayes_textbook import visualization as viz
from bayes_textbook.models import BayesianLinearRegression, ridge_solution
from bayes_textbook.simulation import make_capstone_dataset

pio.renderers.default = "plotly_mimetype+notebook_connected"

x, y = make_capstone_dataset(seed=0)
print(f'{len(x)} points')

40 points


## 事後平均 = リッジ解

ガウス事前 × ガウス尤度 → ガウス事後。閉形式は

$$\Sigma = \left(\tfrac{1}{\sigma^2}\Phi^\top\Phi + \tfrac{1}{\sigma_w^2} I\right)^{-1}, \quad\mu = \tfrac{1}{\sigma^2}\Sigma\,\Phi^\top y.$$

$\lambda = \sigma^2/\sigma_w^2$ と置くと $\mu$ はリッジ解そのもの。確かめる。

In [2]:
def poly_features(x, degree, stats=None):
    X = np.vander(np.asarray(x, dtype=float), degree + 1, increasing=True)
    if stats is None:
        mu, sd = X[:, 1:].mean(0), X[:, 1:].std(0)
    else:
        mu, sd = stats
    Xs = X.copy()
    Xs[:, 1:] = (X[:, 1:] - mu) / sd
    return Xs, (mu, sd)

degree = 12
Phi, stats = poly_features(x, degree)
sigma, sigma_w = 1.0, 1.0
lam = sigma**2 / sigma_w**2
blr = BayesianLinearRegression(sigma=sigma, sigma_w=sigma_w).fit(Phi, y)
w_ridge = ridge_solution(Phi, y, lam)
print(f'lambda = sigma^2 / sigma_w^2 = {lam:g}')
print(f'max |posterior mean - ridge| = {np.max(np.abs(blr.w_mean - w_ridge)):.2e}')
print('-> the LA closed form and the NN gradient-descent result land on this same vector.')

lambda = sigma^2 / sigma_w^2 = 1
max |posterior mean - ridge| = 1.89e-15
-> the LA closed form and the NN gradient-descent result land on this same vector.


## ベイズが足すもの:不確実性

点推定(リッジ解)で止まらず、ベイズは予測の**帯**を返す。データが少ないと帯は広く、増えると締まる。スライダーでデータ件数を増やしてみる(95% 信頼帯=関数の不確実性、95% 予測帯=観測ノイズ込み)。

In [3]:
fig = viz.plotly_posterior_predictive(
    x, y, degree=5, sigma=0.4, sigma_w=5.0, n_frames=10,
    title='Posterior predictive bands tighten as data grows',
)
fig.show()

## 大統一:1つの式、3つのレンズ

$$\underbrace{(\Phi^\top\Phi + \lambda I)^{-1}\Phi^\top y}_{\text{ridge (linear algebra)}}\;=\;\underbrace{\arg\min_w \lVert\Phi w - y\rVert^2 + \lambda\lVert w\rVert^2}_{\text{gradient descent (neural net)}}\;=\;\underbrace{\mathbb{E}[w \mid \Phi, y]}_{\text{posterior mean (Bayes)}}$$

線形代数は**解そのもの**を、ニューラルネットは**解への辿り方**を、ベイズは**解の周りの不確実性**を教える。3冊は同じ構造の別の顔だった。

## 3つのレンズへのリンク

同じデータ・同じ答えを、3冊が別の角度から照らす。

- **線形代数** — `analytics/linear_algebra` 12章「キャップストーン」:正規方程式の閉形式解・SVD・リッジ
- **ニューラルネット** — `analytics/neural_net` 14章「キャップストーン」:勾配降下が同じ解に辿り着く
- **ベイズ** — `analytics/bayesian` 13章「キャップストーン」:重みの事後分布と予測の不確実性

3視点を並べたショーケールは統合ポータル(`analytics/report`、`make report`)の「統合」ページにもある。